# Optimizers: Parameter iteration and iterator exhaustion

**Solution notebook — Delta Drills #445**

Run the cells top-to-bottom to see the reference answer execute.


## Problem

An optimizer iterates over parameters every step, so the params iterator must be materialized as a list first. Build model = torch.nn.Linear(3, 2, bias=False) (set torch.manual_seed(0)) and overwrite its weight with torch.tensor([[1.,2.,3.],[4.,5.,6.]]) under no_grad. Set model.weight.grad = torch.ones_like(model.weight). Capture params = list(model.parameters()). Then run 3 SGD steps with lr=0.1: each step, under no_grad, for p in params do p -= lr * p.grad. Print [round(v,4) for v in model.weight.flatten().tolist()].


<details><summary>💡 Hint (click to reveal)</summary>

The grad is all ones and never recomputed, so each step subtracts the same constant from every weight. Count how many steps run.

</details>


In [ ]:
%pip install -q numpy torch --index-url https://download.pytorch.org/whl/cpu

## Reference solution


In [ ]:
import torch

torch.manual_seed(0)
model = torch.nn.Linear(3, 2, bias=False)
with torch.no_grad():
    model.weight.copy_(torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]]))
model.weight.grad = torch.ones_like(model.weight)

params = list(model.parameters())
lr = 0.1
for step in range(3):
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

print([round(v, 4) for v in model.weight.flatten().tolist()])


## Why this works

Because params is materialized as a list, the same weight tensor is updated on each of the 3 steps. The gradient stays constant at ones, so each step subtracts lr*1=0.1, totaling 0.3 off every weight after 3 steps. The no_grad context lets the in-place subtraction modify a leaf parameter without tracking it.
